# Connecting to TERN STAC API

The generic way is to install `pystac_client`, which is the main python package to interact with STAC catalogs and APIs.

Documentation: https://pystac-client.readthedocs.io/en/stable/

Stac Browser: https://stac-api.tern.org.au/stac-browser/

In [ ]:
!pip install pystac_client
!pip install pandas

## Connect to TERN STAC API

Instantiate a stac client object and point it to TERN STAC API endpoint.

In [1]:
# Import pystac client
from pystac_client import Client
# also import pandas for nicer disply
import pandas as pd

In [2]:
# instantiace client and print info
catalog = Client.open("https://stac-api.tern.org.au/")
print(catalog)

<Client id=stac-fastapi>


## Retrieve List of collections

In [3]:
# get all collections from TERN STAC API
collections = list(catalog.get_collections())
# put collections into pandas dataframe for display
df = pd.DataFrame([
    {
        "ID": c.id,
        "Title": c.title,
        #"Description": c.description,
        #"License": c.license,
        #"Keywords": ", ".join(c.keywords or []),
    }
    for c in collections
]).sort_values("Title").reset_index(drop=True)

# let's show all collections in a table
with pd.option_context("display.max_rows", None):
    display(
    df.style
      .set_properties(**{"text-align": "left", "vertical-align": "top"})
      .set_properties(subset=["ID"], **{"width": "250px"})
      .set_properties(subset=["Title"],**{"width": "100%", "white-space": "normal", "word-wrap": "break-word"})
      .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}])
)

,ID,Title
0,remote-sensing__gedi,"Aboveground Biomass Density - International Space Station, LIDAR, L4A and L4B Models, Australia Coverage, 2020"
1,remote-sensing__gedi_agb_aus_2020_l4a,Aboveground Biomass Density - L4A Models
2,remote-sensing__gedi_agb_aus_2020_l4b,Aboveground Biomass Density - L4B Models
3,model-derived_aet__v2_1,Actual Evapotranspiration for Australia using CMRSET algorithm v2_1
4,model-derived_aet__v2_2,Actual Evapotranspiration for Australia using CMRSET algorithm v2_2
5,gov_qld_burnt_area__qld_annual,"Annual Fire Scars - Landsat, QLD DES algorithm, QLD coverage"
6,gov_qld_burnt_area_landsat__qld_annual,"Annual Landsat Fire Scars - QLD DETSI Algorithm, QLD Coverage"
7,gov_qld_fractional_cover_v3_landsat__aoa,"Area of Applicability Summary, Landsat Seasonal Fractional Cover Version 3.0, Australia Coverage"
8,model-derived_smips_v1_0__smindex,Australia wide daily volumetric soil moisture estimates v1_0 SMindex
9,model-derived_smips_v1_0__bucket1,Australia wide daily volumetric soil moisture estimates v1_0 bucket1


## Access one specific collection by id 

Item 101 - Statewide Landcover and Trees Study (SLATS) Woody Vegetation Change: Regrowth, Clearing and Reports, Queensland - Clearing

id: `gov_qld__woody_vegetation_clearing`

In [4]:
# Access one collection and show some item information
search = catalog.search(
    collections=["gov_qld__woody_vegetation_clearing"]
)
# Get all items in this collection
items = list(search.items())
# Show simple list of items
df = pd.DataFrame([
    {
        "ID": c.id,
        "Date": c.datetime
    }
    for c in items
]).sort_values("Date").reset_index(drop=True)

df

,ID,Date
0,cvmsre_qld_e1819_aera2,2018-01-01 00:00:00+00:00
1,cvmsre_qld_e1920_aera2,2019-01-01 00:00:00+00:00
2,cvmsre_qld_e2021_aera2,2020-01-01 00:00:00+00:00
3,cvmsre_qld_e2122_aera2,2021-01-01 00:00:00+00:00
4,cvmsre_qld_e2223_aera2,2022-01-01 00:00:00+00:00
5,cvmsre_qld_e2324_aera2,2023-01-01 00:00:00+00:00


# Introducing `tern-stac`

A python library that makes it easier to work with TERN STAC API.

`tern-stac` wraps pystac-client and provides convenience loaders for rasterio, xarray, geopandas and LiDAR workflows.

Documentation: https://github.com/ternaustralia/TERN-STAC

PyPI: https://pypi.org/project/tern-stac/


In [ ]:
# install tern-stac
# This will also include all supported integrations like rasterio, xarray, laspy, odc-geo, geopandes, dask, ....
!pip install 'tern-stac[all,odc]'

One of the major advantages of the `tern-stac` library is that it already has the STAC API endpoint preconfigured, and provides some useful helpers / boilerplate to access data directly.

In [6]:
# import and instantiate client
from tern_stac import TernStacClient, load_items_odc
client = TernStacClient()

## List all collection

This shows the same result as above with PySTAC client, not as nicely formatted though.

In [7]:
for c in client.collection_search().collection_list():
    print(c.id)

avhrr_avhrr_toa__v1
curtin_curtin__australian_terrestrial_and_coastal_marine_carbon_stocks_collection
curtin_curtin__soil_carbon_sequestration_collection
edu_curtin__australian_terrestrial_and_coastal_marine_carbon_stocks
edu_curtin__soil_carbon_sequestration
gov_nsw__woody_fpc_extent-2011
gov_qld__foliage_projective_cover
gov_qld__persistent_green_trend_climate
gov_qld__persistent_green_trend_linear
gov_qld__water_count
gov_qld__woody_vegetation_clearing
gov_qld__woody_vegetation_extent
gov_qld__woody_vegetation_regrowth
gov_qld_burnt_area__qld_annual
gov_qld_burnt_area_landsat__qld_annual
gov_qld_burnt_area_landsat__qld_monthly
gov_qld_fire_scars_sentinel2__annual_fire_scars
gov_qld_fire_scars_sentinel2__monthly_fire_scars
gov_qld_fractional_cover_v3_landsat__aoa
gov_qld_fractional_cover_v3_landsat__persistent_green
gov_qld_fractional_cover_v3_landsat__qld
gov_qld_fractional_cover_v3_landsat_fractional_cover__decile_ranking_green
gov_qld_fractional_cover_v3_landsat_fractional_cover__

## Access one collection by id 

In [8]:
# get one collection, This time Fractional Cover dataset
coll = client.get_collection("gov_qld_fractional_cover_v3_landsat_fractional_cover__seasonal")
print("ID:         ", coll.id)
print("Title:      ", coll.title)
print("Description:", coll.description)
print("License:    ", coll.license)

ID:          gov_qld_fractional_cover_v3_landsat_fractional_cover__seasonal
Title:       Seasonal fractional cover - Landsat, JRSRP algorithm Version 3.0, Australia coverage
Description: The seasonal fractional cover product shows representative values for the proportion of bare, green and non-green cover across a season. It is a spatially explicit raster product that predicts vegetation cover at medium resolution (30 m per-pixel) for each 3-month calendar season across Australia from 1987 to the present. The green and non-green fractions may include a mix of woody and non-woody vegetation.A 3 band (byte) image is produced:band 1 - bare ground fraction (in percent),band 2 - green vegetation fraction (in percent),band 3 - non-green vegetation fraction (in percent).The no data value is 255.
License:     CC-BY-4.0


## Get all items for this collection

There are many items in this collection so we don't print the details here this time.

In [9]:
# get all items for this collection and print some basic infos
items = list(coll.get_all_items())
print(len(items))
#for item in items:
#    print(item.id, item.datetime)

154


## Inspect one Item

In [12]:
items[0]

<Item id=lztmre_aus_m202603202605_dp1a2>

## Access this item

Load the VRT (Australia wide coverage) asset into an xarray data structure.

In [ ]:
# Load vrt asset, as Dask chunked array for parallel processing and nicer display in Notebook
d1 = client.load_xarray(items[0], asset_key="lztmre_aus_m202603202605_dp1a2.vrt", chunks=True)
d1

<xarray.DataArray (band: 3, y: 135159, x: 141481)> Size: 57GB
dask.array<open_rasterio-aa7b6c1d0f1cc004ac22340d662223b0<this-array>, shape=(3, 135159, 141481), dtype=uint8, chunksize=(1, 11520, 11520), chunktype=numpy.ndarray>
Coordinates:
  * band         (band) int64 24B 1 2 3
  * y            (y) float64 1MB -8.554e+05 -8.555e+05 ... -4.91e+06 -4.91e+06
  * x            (x) float64 1MB -1.945e+06 -1.945e+06 ... 2.3e+06 2.3e+06
    spatial_ref  int64 8B 0
Attributes:
    _FillValue:    255
    scale_factor:  1.0
    add_offset:    0.0